In [1]:
import pandas as pd
import glob
import os

In [2]:
# === 오브젝트 정의 ===
english_objects = {"Fur ball", "Sponge ball", "BaseBall", "Steal ball", "bouncy ball"}
number_objects = {"0", "10", "30", "50", "100"}
valid_objects = english_objects.union(number_objects)

In [3]:
df = pd.read_csv("./CountGrab/dohoon.csv", encoding="utf-8-sig", header=None, names=["timestamp", "object_name", "object_grab", "total_grab"])

In [4]:
df.head()

,timestamp,object_name,object_grab,total_grab
0,2025-03-26 오전 9:30:39,100,1,1
1,2025-03-26 오전 9:30:42,50,1,2
2,2025-03-26 오전 9:30:45,50,2,3
3,2025-03-26 오전 9:30:48,50,3,4
4,2025-03-26 오전 9:30:51,100,2,5


In [5]:
def search_start_index(df):
    # === 세션 시작 인덱스 찾기 (total_grab == 1)
    session_starts = df.index[df["total_grab"] == 1].tolist()
    session_starts.append(len(df))  # 마지막 세션 범위 끝

    return session_starts

In [6]:
def cut_sessions(df, session_starts):
    # === 한글 AM/PM 처리
    df["timestamp"] = df["timestamp"].astype(str).str.replace("오전", "AM", regex=False)
    df["timestamp"] = df["timestamp"].astype(str).str.replace("오후", "PM", regex=False)

    # === datetime 변환
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", format="%Y-%m-%d %p %I:%M:%S")

    sessions = []
    for i in range(len(session_starts) - 1):
        start_idx = session_starts[i]
        end_idx = session_starts[i + 1]
        session_df = df.iloc[start_idx:end_idx].copy()
        session_df["session"] = i + 1

        # 시간 간격 계산
        start_time = session_df["timestamp"].iloc[0]
        end_time = session_df["timestamp"].iloc[-1]
        duration = (end_time - start_time).total_seconds() if pd.notnull(start_time) and pd.notnull(end_time) else None
        session_df["session_duration"] = duration

        sessions.append(session_df)

    return sessions


In [7]:
def filter_sessions(sessions):
    # === 세션 전체 병합
    df_sessions = pd.concat(sessions, ignore_index=True)


    # === 세션 내 중복 오브젝트 제거 (마지막만 유지)
    df_cleaned = df_sessions.sort_values("timestamp").drop_duplicates(
        subset=["session", "object_name"], keep="last"
    )
    
    return df_cleaned

In [8]:
output_dir = ".\Full\CountGrab_Full\Full_data"
os.makedirs(output_dir, exist_ok=True)

In [10]:
def main(df):
    session_starts = search_start_index(df)
    sessions = cut_sessions(df, session_starts)
    df_cleaned = filter_sessions(sessions)
    return df_cleaned


# === 여러 파일 처리 ===
folder_path = "./CountGrab"
file_list = glob.glob(os.path.join(folder_path, "*.csv"))

for file in file_list:
    file_base = os.path.splitext(os.path.basename(file))[0]
    
    df = pd.read_csv(file, header=None, names=["timestamp", "object_name", "object_grab", "total_grab"])
    df_cleaned = main(df)

    export_path = os.path.join(output_dir, f"{file_base}_full.csv")
    df_cleaned.to_csv(export_path, index=False, encoding='utf-8-sig')
    print(f"✅ {export_path} 저장 완료!")

    # 세션별 행 개수 출력
    print(df_cleaned["session"].value_counts())

✅ .\Full\CountGrab_Full\Full_data\bosung_full.csv 저장 완료!
1     5
2     5
3     5
4     5
5     5
6     5
7     5
8     5
9     5
10    5
Name: session, dtype: int64
✅ .\Full\CountGrab_Full\Full_data\dohoon_full.csv 저장 완료!
1     5
2     5
3     5
4     5
5     5
6     5
7     5
8     5
9     5
10    5
Name: session, dtype: int64
✅ .\Full\CountGrab_Full\Full_data\jaeho_full.csv 저장 완료!
1     5
2     5
3     5
4     5
5     5
6     5
7     5
8     5
9     5
10    5
Name: session, dtype: int64
✅ .\Full\CountGrab_Full\Full_data\jaehu_full.csv 저장 완료!
1     5
2     5
3     5
4     5
5     5
6     5
7     5
8     5
9     5
10    5
Name: session, dtype: int64
✅ .\Full\CountGrab_Full\Full_data\minho_full.csv 저장 완료!
1     5
2     5
3     5
4     5
5     5
6     5
7     5
8     5
9     5
10    5
Name: session, dtype: int64
✅ .\Full\CountGrab_Full\Full_data\minju_full.csv 저장 완료!
1     5
2     5
3     5
4     5
5     5
6     5
7     5
8     5
9     5
10    5
Name: session, dtype: int64
✅ .\Full\Count

In [50]:
file = os.path.join(folder_path, "suyeoun.csv")

In [51]:
import chardet

with open(file, 'rb') as f:
    raw_data = f.read()
    result = chardet.detect(raw_data)
    detected_encoding = result['encoding']
    print(f"{file} → 감지된 인코딩: {detected_encoding}")

df = pd.read_csv(file, encoding=detected_encoding, header=None, names=["timestamp", "object_name", "object_grab", "total_grab"])

./CountGrab\suyeoun.csv → 감지된 인코딩: utf-8


In [52]:
# utf-8-sig로 변환
df.to_csv(file, index=False, encoding='ISO-8859-1', header=False)
print(f"✅ {file} 인코딩 변환 완료!")

UnicodeEncodeError: 'latin-1' codec can't encode characters in position 11-12: ordinal not in range(256)